In [ ]:
# Install required libraries
!pip install -q ultralytics duckduckgo-search bing-image-downloader

import torch
import sys
import os
import shutil
from pathlib import Path

print("Python version :", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device     :", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import zipfile
import pandas as pd

ZIP_PATH = '/content/drive/MyDrive/Tobacco leaf disease detection.v1i.multiclass.zip'
EXTRACT_DIR = '/content/dataset_raw'
YOLO_DIR = '/content/dataset_yolo'

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
if os.path.exists(YOLO_DIR):
    shutil.rmtree(YOLO_DIR)

print(f"Extracting {ZIP_PATH}...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

CLASS_MAP = {
    'alternaria alternata': 'alternaria_alternata',
    'cercospora nicotianae': 'cercospora_nicotianae',
    'no cercospora nicotianae or alternaria alternata present': 'healthy'
}
SPLIT_MAP = {'train': 'train', 'valid': 'val', 'test': 'test'}

def reorganize_split(raw_split, yolo_split):
    raw_dir = os.path.join(EXTRACT_DIR, raw_split)
    csv_path = os.path.join(raw_dir, '_classes.csv')
    if not os.path.exists(csv_path): return
    
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    df['filename'] = df['filename'].str.strip()
    
    for _, folder_name in CLASS_MAP.items():
        os.makedirs(os.path.join(YOLO_DIR, yolo_split, folder_name), exist_ok=True)
        
    for _, row in df.iterrows():
        fname = row['filename']
        src = os.path.join(raw_dir, fname)
        if not os.path.exists(src): continue
        
        class_name = None
        for csv_col, folder in CLASS_MAP.items():
            if csv_col in df.columns and int(row[csv_col]) == 1:
                class_name = folder
                break
        
        if class_name:
            dst = os.path.join(YOLO_DIR, yolo_split, class_name, fname)
            shutil.copy2(src, dst)

for r_split, y_split in SPLIT_MAP.items():
    reorganize_split(r_split, y_split)

print("Existing dataset reorganized into YOLO format!")


In [ ]:
from duckduckgo_search import DDGS
from bing_image_downloader import downloader
from PIL import Image, ImageOps
import io
import requests
import random

def scrape_images(query, max_results=60, output_dir='scraped'):
    os.makedirs(output_dir, exist_ok=True)
    downloaded = 0
    urls = []
    
    print(f"\nSearching for '{query}'...")
    try:
        with DDGS() as ddgs:
            results = list(ddgs.images(query, max_results=max_results))
            urls = [r['image'] for r in results]
            print(f"DDGS found {len(urls)} images.")
    except Exception as e:
        print(f"DDGS failed: {e}")
    
    if len(urls) < max_results:
        print(f"Falling back to Bing Image Downloader...")
        bing_dir = os.path.join(output_dir, "bing_temp")
        try:
            downloader.download(query, limit=max_results, output_dir=bing_dir, adult_filter_off=True, force_replace=False, timeout=10, verbose=False)
            bing_files = [os.path.join(bing_dir, query, f) for f in os.listdir(os.path.join(bing_dir, query))]
            for f in bing_files:
                if downloaded >= max_results: break
                try:
                    img = Image.open(f).convert("RGB")
                    img = ImageOps.exif_transpose(img)
                    img = img.resize((640, 640), Image.Resampling.BILINEAR)
                    dst = os.path.join(output_dir, f"img_{downloaded}.jpg")
                    img.save(dst, "JPEG")
                    downloaded += 1
                except: pass
            shutil.rmtree(bing_dir, ignore_errors=True)
        except Exception as e:
            print(f"Bing downloader failed: {e}")
            
    # Download DDGS URLs if any
    for url in urls:
        if downloaded >= max_results: break
        try:
            r = requests.get(url, timeout=5)
            img = Image.open(io.BytesIO(r.content)).convert("RGB")
            img = ImageOps.exif_transpose(img)
            img = img.resize((640, 640), Image.Resampling.BILINEAR)
            dst = os.path.join(output_dir, f"img_{downloaded}.jpg")
            img.save(dst, "JPEG")
            downloaded += 1
        except:
            continue
            
    print(f"Successfully processed {downloaded} images for '{query}'.")
    return [os.path.join(output_dir, f"img_{i}.jpg") for i in range(downloaded) if os.path.exists(os.path.join(output_dir, f"img_{i}.jpg"))]


In [ ]:
NEW_CLASSES = {
    'tmv': 'Tobacco Mosaic Virus leaf',
    'angular_leaf_spot': 'Tobacco Angular Leaf Spot'
}

for class_name, query in NEW_CLASSES.items():
    raw_images = scrape_images(query, max_results=70, output_dir=f'/content/scraped_{class_name}')
    
    # Shuffle and split
    random.shuffle(raw_images)
    total = len(raw_images)
    train_end = int(total * 0.8)
    val_end = int(total * 0.9)
    
    splits = {
        'train': raw_images[:train_end],
        'val': raw_images[train_end:val_end],
        'test': raw_images[val_end:]
    }
    
    for split_name, imgs in splits.items():
        dst_dir = os.path.join(YOLO_DIR, split_name, class_name)
        os.makedirs(dst_dir, exist_ok=True)
        for idx, img_path in enumerate(imgs):
            dst_path = os.path.join(dst_dir, f"{class_name}_{idx}.jpg")
            shutil.copy2(img_path, dst_path)
            
print("\nAll data ready in YOLO_DIR!")


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n-cls.pt')

results = model.train(
    data=YOLO_DIR,
    epochs=50,
    imgsz=224,
    batch=32,
    patience=10,
    optimizer='AdamW',
    lr0=0.001,
    project='/content/runs/classify',
    name='tobacco_disease_cls_v2'
)

print("Training Complete!")


In [ ]:
best_weights = '/content/runs/classify/tobacco_disease_cls_v2/weights/best.pt'
dst_drive = '/content/drive/MyDrive/tobacco_disease_best_v2.pt'

if os.path.exists(best_weights):
    shutil.copy2(best_weights, dst_drive)
    print(f"Model saved to Google Drive: {dst_drive}")
else:
    print("Error: best.pt not found!")
